# `preprocessing2.py` — Testing & Validation

This notebook tests every step of the preprocessing pipeline:
1. `build_community_timestep_panel()` — community × 15-min aggregation
2. `build_alpha_dataset()` — full (X, y) matrix with alpha target
3. Manual validation of alpha, lags, snapshots, and calendar features
4. Null/missingness analysis across feature columns
5. Leakage sanity checks

In [0]:
from pyspark.sql import functions as F, Window
from preprocessing2 import (
    build_community_timestep_panel,
    build_alpha_dataset,
    FEATURE_COLS,
    CATEGORICAL_COLS,
)

raw_df = spark.table("datathon.shared.client_consumption")
demand = spark.table("datathon.shared.demand_forecast")

print(f"raw_df rows:  {raw_df.count():,}")
print(f"raw_df cols:  {raw_df.columns}")
print(f"demand rows:  {demand.count():,}")
print(f"demand cols:  {demand.columns}")
print(f"\nFEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"CATEGORICAL_COLS ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")

raw_df rows:  182,198,184
raw_df cols:  ['client_id', 'datetime_utc', 'datetime_local', 'community_code', 'active_kw']
demand rows:  10,176
demand cols:  ['datetime_utc', 'datetime_local', 'value']

FEATURE_COLS (24): ['alpha_lag_2d', 'alpha_lag_7d', 'alpha_lag_14d', 'alpha_lag_21d', 'alpha_lag_28d', 'alpha_lag_mean', 'sn_alpha_last', 'sn_alpha_mean_24h', 'sn_alpha_mean_7d', 'sn_alpha_std_7d', 'n_clients_at_cutoff', 'D_mw', 'hour', 'qhour', 'mod', 'dow', 'month', 'doy', 'is_weekend', 'sin_mod', 'cos_mod', 'sin_dow', 'cos_dow', 'community_code']
CATEGORICAL_COLS (6): ['community_code', 'hour', 'qhour', 'dow', 'month', 'is_weekend']


---
## Step 1 — `build_community_timestep_panel()`
Aggregates raw client data into a `(community_code, datetime_utc)` panel with statistics.

In [0]:
panel = build_community_timestep_panel(raw_df)

print(f"Panel rows:   {panel.count():,}")
print(f"Panel cols:   {len(panel.columns)}")
print(f"Columns:      {panel.columns}")
print()

# Expected: 18 communities × 96 slots/day × ~334 days ≈ 576,000+
communities = panel.select("community_code").distinct().count()
time_slots  = panel.select("datetime_utc").distinct().count()
print(f"Distinct communities: {communities}")
print(f"Distinct time slots:  {time_slots:,}")
print(f"communities × slots:  {communities * time_slots:,}  (vs actual {panel.count():,})")
print()

panel.printSchema()
display(panel.orderBy("community_code", "datetime_utc").limit(20))

Panel rows:   537,792
Panel cols:   15
Columns:      ['community_code', 'datetime_utc', 'datetime_local', 'n_clients', 'n_missing', 'n_new_clients', 'total_kw', 'mean_kw', 'median_kw', 'std_kw', 'min_kw', 'max_kw', 'n_reporting', 'avg_kw_per_client', 'avg_kw_reporting']

Distinct communities: 18
Distinct time slots:  32,060
communities × slots:  577,080  (vs actual 537,792)

root
 |-- community_code: string (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- datetime_local: timestamp (nullable = true)
 |-- n_clients: long (nullable = false)
 |-- n_missing: long (nullable = true)
 |-- n_new_clients: long (nullable = true)
 |-- total_kw: double (nullable = true)
 |-- mean_kw: double (nullable = true)
 |-- median_kw: double (nullable = true)
 |-- std_kw: double (nullable = true)
 |-- min_kw: double (nullable = true)
 |-- max_kw: double (nullable = true)
 |-- n_reporting: long (nullable = true)
 |-- avg_kw_per_client: double (nullable = true)
 |-- avg_kw_reporting: double

community_code,datetime_utc,datetime_local,n_clients,n_missing,n_new_clients,total_kw,mean_kw,median_kw,std_kw,min_kw,max_kw,n_reporting,avg_kw_per_client,avg_kw_reporting
AN,2024-12-31T23:00:00.000Z,2025-01-01T00:00:00.000Z,802,0,802,12213.693743999995,15.229044568578548,2.127928,52.670959682761115,0.0,1009.000992,802,15.229044568578548,15.229044568578548
AN,2024-12-31T23:15:00.000Z,2025-01-01T00:15:00.000Z,802,0,0,12213.693743999998,15.229044568578551,2.1312,52.66982194497892,0.0,1009.000992,802,15.229044568578551,15.229044568578551
AN,2024-12-31T23:30:00.000Z,2025-01-01T00:30:00.000Z,802,0,0,12203.811999999996,15.216723192019945,2.0,52.6542524124054,0.0,1008.0,802,15.216723192019945,15.216723192019945
AN,2024-12-31T23:45:00.000Z,2025-01-01T00:45:00.000Z,802,0,0,12208.048512000001,15.222005625935164,2.0,52.61734420708865,0.0,1005.998016,802,15.222005625935164,15.222005625935164
AN,2025-01-01T00:00:00.000Z,2025-01-01T01:00:00.000Z,802,0,0,12107.690568,15.096871032418953,2.0,51.48435188257667,0.0,1004.004004,802,15.096871032418953,15.096871032418953
AN,2025-01-01T00:15:00.000Z,2025-01-01T01:15:00.000Z,802,0,0,12106.010196000001,15.094775805486286,2.0,51.44906800470249,0.0,1002.002004,802,15.094775805486286,15.094775805486286
AN,2025-01-01T00:30:00.000Z,2025-01-01T01:30:00.000Z,802,0,0,12077.814811999995,15.059619466334158,2.0,51.21609420997361,0.0,998.999,802,15.059619466334158,15.059619466334158
AN,2025-01-01T00:45:00.000Z,2025-01-01T01:45:00.000Z,802,0,0,12047.104424,15.021327211970075,2.0,50.79334694030824,0.0,994.994992,802,15.021327211970075,15.021327211970075
AN,2025-01-01T01:00:00.000Z,2025-01-01T02:00:00.000Z,802,0,0,11777.568348000003,14.685247316708233,2.0,49.41257033533461,0.0,986.99088,802,14.685247316708233,14.685247316708233
AN,2025-01-01T01:15:00.000Z,2025-01-01T02:15:00.000Z,802,0,0,11761.056207999996,14.66465861346633,2.003,49.05405970608868,0.0,983.00304,802,14.66465861346633,14.66465861346633


In [0]:
# ─── Pick a specific (community, timestamp) and verify manually ───
SAMPLE_COMM = "AN"
SAMPLE_TS   = "2025-06-15 10:00:00"  # UTC

# Raw data for that slot
raw_slice = raw_df.filter(
    (F.col("community_code") == SAMPLE_COMM) &
    (F.col("datetime_utc") == SAMPLE_TS)
)
raw_count     = raw_slice.count()
raw_total     = raw_slice.select(F.sum("active_kw")).collect()[0][0]
raw_mean      = raw_slice.select(F.mean("active_kw")).collect()[0][0]
raw_n_missing = raw_slice.filter(F.col("active_kw").isNull()).count()

print(f"=== Raw df: community={SAMPLE_COMM}, datetime_utc={SAMPLE_TS} ===")
print(f"  n_clients (distinct client_id): {raw_slice.select('client_id').distinct().count()}")
print(f"  n_rows:     {raw_count}")
print(f"  total_kw:   {raw_total}")
print(f"  mean_kw:    {raw_mean}")
print(f"  n_missing:  {raw_n_missing}")

# Panel row for the same slot
panel_row = panel.filter(
    (F.col("community_code") == SAMPLE_COMM) &
    (F.col("datetime_utc") == SAMPLE_TS)
)
print(f"\n=== Panel row ===")
panel_row.show(truncate=False)

# Compare
if panel_row.count() == 1:
    pr = panel_row.collect()[0]
    print(f"\n--- Comparison ---")
    print(f"  n_clients:  panel={pr['n_clients']}  vs  raw={raw_slice.select('client_id').distinct().count()}")
    print(f"  total_kw:   panel={pr['total_kw']:.4f}  vs  raw={raw_total:.4f}  match={abs(pr['total_kw'] - raw_total) < 0.01}")
    print(f"  mean_kw:    panel={pr['mean_kw']:.4f}  vs  raw={raw_mean:.4f}  match={abs(pr['mean_kw'] - raw_mean) < 0.01}")
    print(f"  n_missing:  panel={pr['n_missing']}  vs  raw={raw_n_missing}  match={pr['n_missing'] == raw_n_missing}")
    print(f"  n_reporting: panel={pr['n_reporting']}  expected={pr['n_clients'] - pr['n_missing']}  match={pr['n_reporting'] == pr['n_clients'] - pr['n_missing']}")
else:
    print("WARNING: Expected exactly 1 panel row, got", panel_row.count())

=== Raw df: community=AN, datetime_utc=2025-06-15 10:00:00 ===
  n_clients (distinct client_id): 1071
  n_rows:     1071
  total_kw:   70816.251628
  mean_kw:    66.1216168328665
  n_missing:  0

=== Panel row ===
+--------------+-------------------+-------------------+---------+---------+-------------+-----------------+-----------------+---------+------------------+------+-------+-----------+-----------------+-----------------+
|community_code|datetime_utc       |datetime_local     |n_clients|n_missing|n_new_clients|total_kw         |mean_kw          |median_kw|std_kw            |min_kw|max_kw |n_reporting|avg_kw_per_client|avg_kw_reporting |
+--------------+-------------------+-------------------+---------+---------+-------------+-----------------+-----------------+---------+------------------+------+-------+-----------+-----------------+-----------------+
|AN            |2025-06-15 10:00:00|2025-06-15 12:00:00|1071     |0        |0            |70816.25162799998|66.12161683286647|4.8

---
## Step 2 — `build_alpha_dataset()`
Full (X, y) matrix. One row per `(community_code, target 15-min slot)`.

In [0]:
X = build_alpha_dataset(raw_df, demand)

print(f"X rows:    {X.count():,}")
print(f"X cols:    {len(X.columns)}")
print(f"Columns:   {X.columns}")
print()

# Date range
X.select(
    F.min("target_date").alias("min_date"),
    F.max("target_date").alias("max_date"),
    F.countDistinct("community_code").alias("n_communities"),
    F.countDistinct("target_date").alias("n_days"),
).show(truncate=False)

print("\nSchema:")
X.printSchema()

print("\nSample rows (AN, 2025-06-15):")
display(
    X.filter((F.col("community_code") == "AN") & (F.col("target_date") == "2025-06-15"))
     .orderBy("datetime_utc")
     .limit(20)
)

X rows:    489,408
X cols:    30
Columns:   ['community_code', 'target_date', 'datetime_utc', 'datetime_local', 'total_kw', 'n_clients', 'y', 'alpha_lag_2d', 'alpha_lag_7d', 'alpha_lag_14d', 'alpha_lag_21d', 'alpha_lag_28d', 'alpha_lag_mean', 'sn_alpha_last', 'sn_alpha_mean_24h', 'sn_alpha_mean_7d', 'sn_alpha_std_7d', 'n_clients_at_cutoff', 'D_mw', 'hour', 'qhour', 'mod', 'dow', 'month', 'doy', 'is_weekend', 'sin_mod', 'cos_mod', 'sin_dow', 'cos_dow']

+----------+----------+-------------+------+
|min_date  |max_date  |n_communities|n_days|
+----------+----------+-------------+------+
|2025-01-29|2025-11-30|18           |306   |
+----------+----------+-------------+------+


Schema:
root
 |-- community_code: string (nullable = true)
 |-- target_date: date (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- datetime_local: timestamp (nullable = true)
 |-- total_kw: double (nullable = true)
 |-- n_clients: long (nullable = false)
 |-- y: double (nullable = true)
 |-- al

community_code,target_date,datetime_utc,datetime_local,total_kw,n_clients,y,alpha_lag_2d,alpha_lag_7d,alpha_lag_14d,alpha_lag_21d,alpha_lag_28d,alpha_lag_mean,sn_alpha_last,sn_alpha_mean_24h,sn_alpha_mean_7d,sn_alpha_std_7d,n_clients_at_cutoff,D_mw,hour,qhour,mod,dow,month,doy,is_weekend,sin_mod,cos_mod,sin_dow,cos_dow
AN,2025-06-15,2025-06-14T22:00:00.000Z,2025-06-15T00:00:00.000Z,68069.891432,1071,0.0026181673538549595,0.0025186284714603926,0.0025679935359621767,0.002583096904707113,0.001911454056590411,0.0010318897312007142,0.0021226125399841614,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,24275.5,0,0,0,1,6,166,1,0.0,1.0,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T22:15:00.000Z,2025-06-15T00:15:00.000Z,69289.891432,1071,0.002665092126386644,0.0025091545640010893,0.002547548031780396,0.002588448130967171,0.001902481651420432,0.0010290603627630531,0.002115338548186428,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,24275.5,0,1,1,1,6,166,1,0.06540312923014306,0.9978589232386035,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T22:30:00.000Z,2025-06-15T00:30:00.000Z,68557.71299999999,1071,0.0026369303998504097,0.0024971719309653598,0.0025231230865434,0.0025717323715605854,0.0018735563361596925,0.0010063815963022632,0.0020943930643062604,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,24275.5,0,2,2,1,6,166,1,0.13052619222005157,0.9914448613738104,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T22:45:00.000Z,2025-06-15T00:45:00.000Z,67837.356136,1071,0.002609223365436609,0.002470909960055278,0.0025113306723988835,0.002545383358091608,0.0018684845595792665,9.972753464882163E-4,0.0020786767793226505,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,24275.5,0,3,3,1,6,166,1,0.19509032201612825,0.9807852804032304,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T23:00:00.000Z,2025-06-15T01:00:00.000Z,66611.318236,1071,0.002727116711771695,0.0025601957646717174,0.002612401768328634,0.002642039172779352,0.0019530198613829921,0.0010186469563401395,0.002157260704700567,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,22806.3,1,0,4,1,6,166,1,0.25881904510252074,0.9659258262890683,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T23:15:00.000Z,2025-06-15T01:15:00.000Z,66012.408288,1071,0.002702596894850336,0.0025367760348966126,0.0025816509813855145,0.0026242482131996507,0.0019366289437558698,9.923090631418687E-4,0.002134322647275903,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,22806.3,1,1,5,1,6,166,1,0.3214394653031616,0.9469301294951057,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T23:30:00.000Z,2025-06-15T01:30:00.000Z,65216.95204000002,1071,0.00267003032681278,0.0025147205138598776,0.002557672669653684,0.002600091974301195,0.0019016232082553657,9.716222003861657E-4,0.0021091461132912576,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,22806.3,1,2,6,1,6,166,1,0.3826834323650898,0.9238795325112867,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-14T23:45:00.000Z,2025-06-15T01:45:00.000Z,65164.949435999995,1071,0.002667901301683422,0.0024990204867859638,0.002557351676085084,0.00257223743812045,0.0018748780867458967,9.386055722213516E-4,0.0020884186519917495,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8018067880427903E-4,1071,22806.3,1,3,7,1,6,166,1,0.44228869021900125,0.8968727415326884,0.7818314824680298,0.6234898018587336
AN,2025-06-15,2025-06-15T00:00:00.000Z,2025-06-15T02:00:00.000Z,67131.02454400001,1071,0.0028826264899678385,0.002580014543107904,0.0026415878923891894,0.002775960847464034,0.0019383190429136954,9.681668785996242E-4,0.00218080984089489,0.0028651009473754704,0.0026761866979489665,0.00258820055812331,1.8

In [0]:
# ─── Manually verify alpha for a few rows ───
print("=== Alpha validation: alpha should equal total_kw / (n_clients * D_mw) ===")

sample = (
    X.filter(F.col("y").isNotNull() & F.col("D_mw").isNotNull())
     .select("community_code", "datetime_utc", "total_kw", "n_clients", "D_mw", "y")
     .withColumn("alpha_manual", F.col("total_kw") / (F.col("n_clients") * F.col("D_mw")))
     .withColumn("alpha_diff", F.abs(F.col("y") - F.col("alpha_manual")))
     .orderBy("community_code", "datetime_utc")
     .limit(20)
)

display(sample)

# Check all rows match
mismatch_count = (
    X.filter(F.col("y").isNotNull() & F.col("D_mw").isNotNull())
     .withColumn("alpha_manual", F.col("total_kw") / (F.col("n_clients") * F.col("D_mw")))
     .filter(F.abs(F.col("y") - F.col("alpha_manual")) > 1e-6)
     .count()
)
print(f"\nRows where alpha ≠ total_kw/(n_clients*D_mw): {mismatch_count}  {'✅ PASS' if mismatch_count == 0 else '❌ FAIL'}")

=== Alpha validation: alpha should equal total_kw / (n_clients * D_mw) ===


community_code,datetime_utc,total_kw,n_clients,D_mw,y,alpha_manual,alpha_diff
AN,2025-01-28T23:00:00.000Z,26166.75669999999,818,26492.0,0.0012074852831892413,0.0012074852831892413,0.0
AN,2025-01-28T23:15:00.000Z,26170.756699999998,818,26492.0,0.0012076698662916922,0.0012076698662916922,0.0
AN,2025-01-28T23:30:00.000Z,26146.395999999997,818,26492.0,0.0012065457228957249,0.0012065457228957249,0.0
AN,2025-01-28T23:45:00.000Z,26117.6746,818,26492.0,0.0012052203516160435,0.0012052203516160435,0.0
AN,2025-01-29T00:00:00.000Z,25591.075004000006,818,24847.3,0.0012590878191081594,0.0012590878191081594,0.0
AN,2025-01-29T00:15:00.000Z,25577.439312,818,24847.3,0.0012584169393698274,0.0012584169393698274,0.0
AN,2025-01-29T00:30:00.000Z,25554.630168,818,24847.3,0.0012572947233093377,0.0012572947233093377,0.0
AN,2025-01-29T00:45:00.000Z,25546.647516,818,24847.3,0.001256901974669595,0.001256901974669595,0.0
AN,2025-01-29T01:00:00.000Z,25345.841684,818,23810.8,0.001301305984385545,0.001301305984385545,0.0
AN,2025-01-29T01:15:00.000Z,25313.361804000007,818,23810.8,0.001299638402667681,0.001299638402667681,0.0



Rows where alpha ≠ total_kw/(n_clients*D_mw): 0  ✅ PASS


In [0]:
# ─── Verify alpha_lag_Nd matches alpha from N days earlier (same community, same UTC time) ───
print("=== Lag validation: alpha_lag_Nd should equal alpha at (datetime_utc - N days) ===")

# Use a small slice for manual check
SAMPLE_COMM = "CT"
SAMPLE_DATE = "2025-07-10"  # Target date

slice_rows = (
    X.filter((F.col("community_code") == SAMPLE_COMM) & (F.col("target_date") == SAMPLE_DATE))
     .orderBy("datetime_utc")
     .select("community_code", "datetime_utc", "datetime_local", "y",
             "alpha_lag_2d", "alpha_lag_7d", "alpha_lag_14d")
     .limit(5)
     .collect()
)

for row in slice_rows:
    ts = row["datetime_utc"]
    print(f"\n--- {SAMPLE_COMM} @ {ts} (local: {row['datetime_local']}) ---")
    print(f"  y (alpha):        {row['y']:.6f}")

    for N, col in [(2, "alpha_lag_2d"), (7, "alpha_lag_7d"), (14, "alpha_lag_14d")]:
        # Look up the original alpha N days before
        shifted_ts = ts - __import__('datetime').timedelta(days=N)
        original = (
            X.filter((F.col("community_code") == SAMPLE_COMM) &
                     (F.col("datetime_utc") == shifted_ts))
             .select("y").collect()
        )
        original_val = original[0]["y"] if original else None
        lag_val = row[col]
        match = (original_val is None and lag_val is None) or \
                (original_val is not None and lag_val is not None and abs(original_val - lag_val) < 1e-6)
        print(f"  {col}: {lag_val}  vs  alpha@{shifted_ts}: {original_val}  {'✅' if match else '❌'}")

=== Lag validation: alpha_lag_Nd should equal alpha at (datetime_utc - N days) ===

--- CT @ 2025-07-09 22:00:00 (local: 2025-07-10 00:00:00) ---
  y (alpha):        0.001761
  alpha_lag_2d: 0.001820941227613924  vs  alpha@2025-07-07 22:00:00: 0.0018209412276139225  ✅
  alpha_lag_7d: 0.0017958596319712977  vs  alpha@2025-07-02 22:00:00: 0.0017958596319712977  ✅
  alpha_lag_14d: 0.0019217576969760215  vs  alpha@2025-06-25 22:00:00: 0.0019217576969760208  ✅

--- CT @ 2025-07-09 22:15:00 (local: 2025-07-10 00:15:00) ---
  y (alpha):        0.001743
  alpha_lag_2d: 0.001800122527096733  vs  alpha@2025-07-07 22:15:00: 0.001800122527096732  ✅
  alpha_lag_7d: 0.0017878370155738643  vs  alpha@2025-07-02 22:15:00: 0.0017878370155738658  ✅
  alpha_lag_14d: 0.0018763246250359004  vs  alpha@2025-06-25 22:15:00: 0.0018763246250359  ✅

--- CT @ 2025-07-09 22:30:00 (local: 2025-07-10 00:30:00) ---
  y (alpha):        0.001752
  alpha_lag_2d: 0.0017800324259792813  vs  alpha@2025-07-07 22:30:00: 0.001

In [0]:
# ─── Snapshot features should come from D-1 at 11:45 local ───
print("=== Snapshot validation: features come from D-1 @ 11:45 local ===")

SAMPLE_COMM = "PV"
TARGET_DATE = "2025-08-20"

# Get snapshot features from X
snap_in_X = (
    X.filter((F.col("community_code") == SAMPLE_COMM) & (F.col("target_date") == TARGET_DATE))
     .select("sn_alpha_last", "sn_alpha_mean_24h", "sn_alpha_mean_7d",
             "sn_alpha_std_7d", "n_clients_at_cutoff")
     .limit(1).collect()
)

if snap_in_X:
    print(f"  sn_alpha_last:       {snap_in_X[0]['sn_alpha_last']}")
    print(f"  sn_alpha_mean_24h:   {snap_in_X[0]['sn_alpha_mean_24h']}")
    print(f"  sn_alpha_mean_7d:    {snap_in_X[0]['sn_alpha_mean_7d']}")
    print(f"  sn_alpha_std_7d:     {snap_in_X[0]['sn_alpha_std_7d']}")
    print(f"  n_clients_at_cutoff: {snap_in_X[0]['n_clients_at_cutoff']}")

    # Verify: the snapshot should come from 2025-08-19 @ 11:45 local
    # Rebuild panel+alpha to check
    snap_direct = (
        panel.withColumn("_h", F.date_trunc("hour", F.col("datetime_utc")))
             .join(
                 demand.select(
                     F.date_trunc("hour", F.col("datetime_utc")).alias("_h"),
                     F.col("value").alias("D_mw")),
                 "_h", "left")
             .withColumn("alpha",
                 F.when((F.col("n_clients") > 0) & (F.col("D_mw") > 0) & F.col("total_kw").isNotNull(),
                        F.col("total_kw") / (F.col("n_clients") * F.col("D_mw"))))
             .filter(
                 (F.col("community_code") == SAMPLE_COMM) &
                 (F.to_date("datetime_local") == "2025-08-19") &
                 (F.hour("datetime_local") == 11) &
                 (F.minute("datetime_local") == 45))
             .select("datetime_local", "alpha", "n_clients")
             .collect()
    )
    if snap_direct:
        print(f"\n  Direct lookup at 2025-08-19 11:45 local:")
        print(f"    alpha:     {snap_direct[0]['alpha']}")
        print(f"    n_clients: {snap_direct[0]['n_clients']}")
        sn_match = (snap_in_X[0]['sn_alpha_last'] is not None and
                    snap_direct[0]['alpha'] is not None and
                    abs(snap_in_X[0]['sn_alpha_last'] - snap_direct[0]['alpha']) < 1e-6)
        nc_match = snap_in_X[0]['n_clients_at_cutoff'] == snap_direct[0]['n_clients']
        print(f"    sn_alpha_last match:       {'✅' if sn_match else '❌'}")
        print(f"    n_clients_at_cutoff match: {'✅' if nc_match else '❌'}")
else:
    print(f"  No rows found for {SAMPLE_COMM} on {TARGET_DATE}")

=== Snapshot validation: features come from D-1 @ 11:45 local ===
  sn_alpha_last:       0.00398214291903503
  sn_alpha_mean_24h:   0.003498317198171636
  sn_alpha_mean_7d:    0.003984460228622047
  sn_alpha_std_7d:     0.0008037466951493326
  n_clients_at_cutoff: 199

  Direct lookup at 2025-08-19 11:45 local:
    alpha:     0.00398214291903503
    n_clients: 199
    sn_alpha_last match:       ✅
    n_clients_at_cutoff match: ✅


In [0]:
# ─── Spot-check calendar features ───
print("=== Calendar feature validation ===")
import datetime

check_rows = (
    X.filter(F.col("community_code") == "MD")
     .select("datetime_local", "hour", "qhour", "mod", "dow", "month",
             "doy", "is_weekend", "sin_mod", "cos_mod")
     .orderBy("datetime_local")
     .limit(10)
     .collect()
)

import math
TAU = 2 * math.pi

all_ok = True
for row in check_rows:
    dt = row["datetime_local"]
    exp_hour    = dt.hour
    exp_qhour   = dt.minute // 15
    exp_mod     = exp_hour * 4 + exp_qhour
    exp_dow     = dt.isoweekday() % 7 + 1  # Spark dayofweek: Sun=1, Mon=2, ..., Sat=7
    exp_month   = dt.month
    exp_doy     = dt.timetuple().tm_yday
    exp_weekend = 1 if exp_dow in (1, 7) else 0
    exp_sin_mod = math.sin(TAU * exp_mod / 96)
    exp_cos_mod = math.cos(TAU * exp_mod / 96)

    checks = {
        "hour":       row["hour"] == exp_hour,
        "qhour":      row["qhour"] == exp_qhour,
        "mod":        row["mod"] == exp_mod,
        "dow":        row["dow"] == exp_dow,
        "month":      row["month"] == exp_month,
        "doy":        row["doy"] == exp_doy,
        "is_weekend": row["is_weekend"] == exp_weekend,
        "sin_mod":    abs(row["sin_mod"] - exp_sin_mod) < 1e-6,
        "cos_mod":    abs(row["cos_mod"] - exp_cos_mod) < 1e-6,
    }
    failed = [k for k, v in checks.items() if not v]
    if failed:
        all_ok = False
        print(f"  ❌ {dt}: failed on {failed}")
        print(f"     actual: {dict((k, row[k]) for k in failed)}")

print(f"\n{'✅ All calendar features correct' if all_ok else '❌ Some mismatches found'}")

=== Calendar feature validation ===

✅ All calendar features correct


In [0]:
# ─── Check nulls across all feature columns and label ───
print("=== Null analysis across features ===")

total_rows = X.count()
print(f"Total rows in X: {total_rows:,}\n")

null_stats = []
for col in ["y"] + FEATURE_COLS:
    if col in X.columns:
        n_null = X.filter(F.col(col).isNull()).count()
        null_stats.append((col, n_null, n_null / total_rows * 100))

null_df = spark.createDataFrame(
    null_stats, schema=["column", "n_nulls", "null_pct"]
).orderBy(F.desc("null_pct"))

display(null_df)

# Rows where ALL lags are null (not enough history)
all_lags_null = (
    X.filter(
        F.col("alpha_lag_2d").isNull() &
        F.col("alpha_lag_7d").isNull() &
        F.col("alpha_lag_14d").isNull() &
        F.col("alpha_lag_21d").isNull() &
        F.col("alpha_lag_28d").isNull()
    ).count()
)
print(f"\nRows with ALL lag features null: {all_lags_null:,}  ({all_lags_null/total_rows*100:.2f}%)")

# Rows where snapshot is null
snap_null = X.filter(F.col("sn_alpha_last").isNull()).count()
print(f"Rows with snapshot features null: {snap_null:,}  ({snap_null/total_rows*100:.2f}%)")

# Rows where D_mw is null
d_null = X.filter(F.col("D_mw").isNull()).count()
print(f"Rows with D_mw null: {d_null:,}  ({d_null/total_rows*100:.2f}%)")

=== Null analysis across features ===
Total rows in X: 489,408



column,n_nulls,null_pct
alpha_lag_2d,104,0.021250163462795867
alpha_lag_7d,104,0.021250163462795867
alpha_lag_14d,104,0.021250163462795867
alpha_lag_21d,104,0.021250163462795867
alpha_lag_28d,104,0.021250163462795867
y,0,0.0
alpha_lag_mean,0,0.0
sn_alpha_last,0,0.0
sn_alpha_mean_24h,0,0.0
sn_alpha_mean_7d,0,0.0



Rows with ALL lag features null: 0  (0.00%)
Rows with snapshot features null: 0  (0.00%)
Rows with D_mw null: 0  (0.00%)


In [0]:
# ─── Leakage check: no feature should use data from day D or later ───
print("=== Leakage sanity checks ===")

# CHECK 1: alpha_lag_2d should be at least 2 days before target
# i.e., the datetime of the lagged value is datetime_utc - 2 days
# which is always < D-1 12:00 for any slot on day D
min_lag = 2
check1 = X.filter(F.col("alpha_lag_2d").isNotNull()).count()
print(f"\n1. alpha_lag_2d: {check1:,} non-null rows (min lag = {min_lag} days, safe)")

# CHECK 2: snapshot comes from D-1 (not D)
# All snapshot rows should have target_date = date(snapshot_time) + 1
# We verify by checking n_clients_at_cutoff distribution
print("\n2. Snapshot n_clients_at_cutoff distribution by month:")
X.withColumn("target_month", F.date_format("target_date", "yyyy-MM")) \
  .groupBy("target_month") \
  .agg(
      F.avg("n_clients_at_cutoff").alias("avg_n_clients"),
      F.min("n_clients_at_cutoff").alias("min_n_clients"),
      F.max("n_clients_at_cutoff").alias("max_n_clients"),
      F.sum(F.when(F.col("n_clients_at_cutoff").isNull(), 1).otherwise(0)).alias("n_null"),
  ) \
  .orderBy("target_month").show(truncate=False)

# CHECK 3: D_mw is external forecast — should never be null for dates within
# demand_forecast range (Jan 2025 - Feb 2026)
print("3. D_mw coverage (should be ~0% null within forecast range):")
X.filter(F.col("target_date") <= "2025-11-30") \
 .select(
     F.count("*").alias("total"),
     F.sum(F.when(F.col("D_mw").isNull(), 1).otherwise(0)).alias("D_mw_null")
 ).show()

# CHECK 4: y should never be null (filtered in step 9 of preprocessing)
y_null = X.filter(F.col("y").isNull()).count()
print(f"4. y (label) null count: {y_null}  {'✅ PASS' if y_null == 0 else '❌ FAIL'}")

# CHECK 5: min_history_days filter — no row should exist within 28 days of
# the community's first appearance
first_dates = (
    raw_df.groupBy("community_code")
          .agg(F.min(F.to_date("datetime_local")).alias("first_date"))
)
early_rows = (
    X.join(first_dates, "community_code")
     .filter(F.datediff("target_date", "first_date") < 28)
     .count()
)
print(f"5. Rows with < 28 days history: {early_rows}  {'✅ PASS' if early_rows == 0 else '❌ FAIL'}")

=== Leakage sanity checks ===

1. alpha_lag_2d: 489,304 non-null rows (min lag = 2 days, safe)

2. Snapshot n_clients_at_cutoff distribution by month:
+------------+------------------+-------------+-------------+------+
|target_month|avg_n_clients     |min_n_clients|max_n_clients|n_null|
+------------+------------------+-------------+-------------+------+
|2025-01     |290.97777777777776|19           |820          |0     |
|2025-02     |294.46190476190475|19           |833          |0     |
|2025-03     |290.44608183079055|1            |858          |0     |
|2025-04     |292.6770833333333 |1            |908          |0     |
|2025-05     |331.9519038076152 |1            |1049         |0     |
|2025-06     |326.6764705882353 |1            |1095         |0     |
|2025-07     |341.8709677419355 |1            |1187         |0     |
|2025-08     |355.2030360531309 |1            |1212         |0     |
|2025-09     |376.5675146771037 |1            |1306         |0     |
|2025-10     |399.949

In [0]:
# ─── Train / Validation split ───
VAL_START = "2025-11-01"

train = X.filter(F.col("target_date") < VAL_START)
val   = X.filter(F.col("target_date") >= VAL_START)

print(f"Train rows: {train.count():,}")
print(f"Val rows:   {val.count():,}")
print(f"Val % :     {val.count() / X.count() * 100:.1f}%")

# Label distribution
print("\n=== Label (y = alpha) distribution ===")
for name, subset in [("Train", train), ("Val", val)]:
    stats = subset.select(
        F.mean("y").alias("mean"),
        F.stddev("y").alias("std"),
        F.min("y").alias("min"),
        F.expr("percentile_approx(y, 0.25)").alias("p25"),
        F.expr("percentile_approx(y, 0.5)").alias("median"),
        F.expr("percentile_approx(y, 0.75)").alias("p75"),
        F.max("y").alias("max"),
    ).collect()[0]
    print(f"  {name:5s}: mean={stats['mean']:.6f}  std={stats['std']:.6f}  "
          f"min={stats['min']:.6f}  p25={stats['p25']:.6f}  "
          f"med={stats['median']:.6f}  p75={stats['p75']:.6f}  max={stats['max']:.6f}")

# Communities in val
print("\n=== Communities in validation ===")
val.groupBy("community_code").agg(
    F.count("*").alias("n_rows"),
    F.avg("y").alias("avg_alpha"),
    F.avg("n_clients_at_cutoff").alias("avg_n_clients"),
    F.avg("D_mw").alias("avg_D_mw"),
).orderBy("community_code").show(20, truncate=False)

print("\n=== Feature correlations with y (train set, sampled) ===")
import pandas as pd
numeric_features = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]
train_sample = train.select("y", *numeric_features).sample(0.05).toPandas()
corr = train_sample.corr()["y"].drop("y").sort_values(ascending=False)
print(corr.to_string())

Train rows: 437,568
Val rows:   51,840
Val % :     10.6%

=== Label (y = alpha) distribution ===
  Train: mean=0.002290  std=0.001700  min=0.000000  p25=0.001182  med=0.001872  p75=0.002927  max=0.014515
  Val  : mean=0.001678  std=0.001107  min=0.000000  p25=0.000815  med=0.001374  p75=0.002180  max=0.007866

=== Communities in validation ===
+--------------+------+---------------------+------------------+------------------+
|community_code|n_rows|avg_alpha            |avg_n_clients     |avg_D_mw          |
+--------------+------+---------------------+------------------+------------------+
|AN            |2880  |7.57653241135449E-4  |1570.7666666666667|26999.268472221433|
|AR            |2880  |6.101062280566106E-4 |425.1             |26999.26847222145 |
|AS            |2880  |0.0025445588027069132|120.33333333333333|26999.26847222145 |
|CB            |2880  |0.0017037503489539586|75.4              |26999.26847222146 |
|CE            |2880  |0.0012884665249964951|1.0               |26